# 17 - Quantum Machine Learning

Build a simple parameterized quantum circuit for binary classification.

**Concepts:** Data encoding, parameterized gates, measurement-based prediction

In [ ]:
import quantsdk as qs
import math
import numpy as np

## Quantum Classifier

1. **Encode** classical data as rotation angles
2. **Process** with parameterized layers
3. **Measure** to get a classification

In [ ]:
def quantum_classifier(x: list, weights: list) -> qs.Circuit:
    """2-qubit quantum classifier.
    
    Args:
        x: Input features [x0, x1].
        weights: Trainable parameters [w0, w1, w2, w3].
    """
    circuit = qs.Circuit(2, name="classifier")
    
    # Data encoding layer
    circuit.ry(0, x[0] * math.pi)
    circuit.ry(1, x[1] * math.pi)
    
    # Variational layer 1
    circuit.ry(0, weights[0])
    circuit.ry(1, weights[1])
    circuit.cx(0, 1)
    
    # Variational layer 2
    circuit.ry(0, weights[2])
    circuit.ry(1, weights[3])
    circuit.cx(1, 0)
    
    circuit.measure(0)  # Classification qubit
    return circuit

def predict(x: list, weights: list, shots: int = 1000) -> float:
    """Return P(class=1) for input x."""
    circuit = quantum_classifier(x, weights)
    result = qs.run(circuit, shots=shots, seed=42)
    p1 = 0
    for bs, count in result.counts.items():
        if bs[0] == '1':
            p1 += count
    return p1 / shots

In [ ]:
# Simple XOR-like dataset
data = [
    ([0.0, 0.0], 0),
    ([0.0, 1.0], 1),
    ([1.0, 0.0], 1),
    ([1.0, 1.0], 0),
]

# Random initial weights
np.random.seed(42)
weights = list(np.random.uniform(0, 2 * math.pi, 4))

print("Before training:")
for x, label in data:
    p = predict(x, weights)
    pred = 1 if p > 0.5 else 0
    print(f"  x={x}, label={label}, P(1)={p:.3f}, pred={pred} {'OK' if pred == label else 'WRONG'}")

In [ ]:
# Simple parameter sweep to improve
def loss(weights: list) -> float:
    """Mean squared error."""
    total = 0.0
    for x, label in data:
        p = predict(x, weights)
        total += (p - label) ** 2
    return total / len(data)

# Coordinate descent
best_w = list(weights)
best_loss = loss(best_w)

for iteration in range(3):
    for i in range(4):
        for delta in np.linspace(-0.5, 0.5, 11):
            trial = list(best_w)
            trial[i] += delta
            l = loss(trial)
            if l < best_loss:
                best_loss = l
                best_w = trial

print(f"Optimized loss: {best_loss:.4f}")
print(f"\nAfter training:")
correct = 0
for x, label in data:
    p = predict(x, best_w)
    pred = 1 if p > 0.5 else 0
    if pred == label: correct += 1
    print(f"  x={x}, label={label}, P(1)={p:.3f}, pred={pred} {'OK' if pred == label else 'WRONG'}")
print(f"Accuracy: {correct}/{len(data)}")